In [ ]:

!pip -q install ipywidgets

import ipywidgets as widgets
from IPython.display import display

print("Середовище для лабораторної роботи готове!")


Середовище для лабораторної роботи готове!


In [ ]:

%%writefile uv_rules.py
import re

VOWELS = "аеєиіїоуюя"
CONSONANTS = "бвгґджзйклмнпрстфхцчшщ"

def check_uv(text):
    """Перевірка окремих правил чергування у/в."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Введіть український текст для перевірки.")

    words = list(re.finditer(r"[А-Яа-яІіЇїЄєҐґ']+", text))
    results = []

    for i, match in enumerate(words):
        word = match.group()
        if word.lower() not in ("у", "в") or i == len(words) - 1:
            continue

        next_word = words[i + 1].group()
        previous_word = words[i - 1].group() if i > 0 else ""

        prev = previous_word[-1].lower() if previous_word else ""
        nxt = next_word[0].lower()

        # Базові правила милозвучності
        suggestion = None
        reason = ""

        if prev in CONSONANTS and nxt in CONSONANTS:
            suggestion = "у"
            reason = "Між приголосними переважно вживаємо «у»."
        elif prev in VOWELS and nxt in VOWELS:
            suggestion = "в"
            reason = "Між голосними переважно вживаємо «в»."
        elif not previous_word and nxt in CONSONANTS:
            suggestion = "у"
            reason = "На початку речення перед приголосним часто вживаємо «у»."

        if suggestion and word.lower() != suggestion:
            results.append({
                "word": word,
                "suggestion": suggestion.upper() if word[0].isupper() else suggestion,
                "reason": reason,
                "position": match.start()
            })

    return results


Overwriting uv_rules.py


In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output
from uv_rules import check_uv

title = widgets.HTML(
    "<h2>Перевірка вживання у/в</h2>"
    "<p>Введіть український текст для аналізу чергування прийменників.</p>"
)

text_input = widgets.Textarea(
    value="Ми зустрілися в школі. В нашому місті багато парків.",
    placeholder="Введіть український текст...",
    description="Текст:",
    layout=widgets.Layout(width="95%", height="150px"),
    style={"description_width": "60px"}
)

check_button = widgets.Button(
    description="Перевірити текст",
    button_style="success",
    icon="check"
)

output = widgets.Output()

def on_check_clicked(button):
    with output:
        clear_output()
        try:
            results = check_uv(text_input.value)

            if not results:
                print("Можливих порушень базових правил не знайдено.")
            else:
                print(f"Знайдено можливих порушень: {len(results)}\n")

                for i, item in enumerate(results, 1):
                    print(f"{i}. {item['word']} → {item['suggestion']}")
                    print(f"   Пояснення: {item['reason']}\n")

        except ValueError as error:
            print(f"Помилка: {error}")
        except Exception as error:
            print(f"Не вдалося перевірити текст: {error}")

check_button.on_click(on_check_clicked)

display(widgets.VBox([
    title,
    text_input,
    check_button,
    output
]))


In [ ]:

tests = {
    "test1.txt": "Ми гуляли в парку. Діти гралися в дворі. Учні навчалися у школі.",
    "test2.txt": "В нашому місті відкрили бібліотеку. В класі було тихо. Всі учні прийшли вчасно.",
    "test3.txt": "Вона живе у Одесі. Ми зустрілися в університеті. Вчитель працює в школі."
}

for filename, content in tests.items():
    with open(filename, "w", encoding="utf-8") as file:
        file.write(content)

print("Успішно створено 3 тестові тексти!")


Успішно створено 3 тестові тексти!


In [6]:

%%writefile test_uv.py
import unittest
from uv_rules import check_uv

class TestUV(unittest.TestCase):

    def test_empty_text(self):
        with self.assertRaises(ValueError):
            check_uv("")

    def test_consonants(self):
        result = check_uv("Ми були в школі.")
        self.assertEqual(len(result), 1)
        self.assertEqual(result[0]["suggestion"], "у")

    def test_vowels(self):
        result = check_uv("Вона була у Одесі.")
        self.assertEqual(len(result), 1)
        self.assertEqual(result[0]["suggestion"], "в")

    def test_correct_text(self):
        result = check_uv("Ми були у школі.")
        self.assertEqual(len(result), 0)

if __name__ == "__main__":
    unittest.main()


Writing test_uv.py


In [7]:

!python -m unittest test_uv.py -v


test_consonants (test_uv.TestUV.test_consonants) ... FAIL
test_correct_text (test_uv.TestUV.test_correct_text) ... ok
test_empty_text (test_uv.TestUV.test_empty_text) ... ok
test_vowels (test_uv.TestUV.test_vowels) ... ok

FAIL: test_consonants (test_uv.TestUV.test_consonants)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/content/test_uv.py", line 12, in test_consonants
    self.assertEqual(len(result), 1)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
AssertionError: 0 != 1

----------------------------------------------------------------------
Ran 4 tests in 0.002s

FAILED (failures=1)


In [8]:

%%writefile uv_rules.py
import re

VOWELS = "аеєиіїоуюя"
CONSONANTS = "бвгґджзйклмнпрстфхцчшщ"

def check_uv(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Введіть український текст.")

    words = list(re.finditer(r"[А-Яа-яІіЇїЄєҐґ']+", text))
    results = []

    for i, match in enumerate(words):
        word = match.group()

        if word.lower() not in ("у", "в") or i == len(words) - 1:
            continue

        next_word = words[i + 1].group()
        previous_word = words[i - 1].group() if i > 0 else ""

        prev = previous_word[-1].lower() if previous_word else ""
        nxt = next_word[0].lower()

        suggestion = None
        reason = ""

        if prev in CONSONANTS and nxt in CONSONANTS:
            suggestion = "у"
            reason = "Між приголосними переважно вживаємо «у»."

        elif prev in VOWELS and nxt in VOWELS:
            suggestion = "в"
            reason = "Між голосними переважно вживаємо «в»."

        elif prev in VOWELS and nxt in CONSONANTS:
            suggestion = "в"
            reason = "Після голосного перед приголосним переважно вживаємо «в»."

        elif not previous_word and nxt in CONSONANTS:
            suggestion = "у"
            reason = "На початку речення перед приголосним переважно вживаємо «у»."

        if suggestion and word.lower() != suggestion:
            results.append({
                "word": word,
                "suggestion": suggestion.upper() if word[0].isupper() else suggestion,
                "reason": reason,
                "position": match.start()
            })

    return results


Overwriting uv_rules.py


In [9]:

%%writefile test_uv.py
import unittest
from uv_rules import check_uv

class TestUV(unittest.TestCase):

    def test_empty_text(self):
        with self.assertRaises(ValueError):
            check_uv("")

    def test_consonants(self):
        result = check_uv("Наш клас в школі.")
        self.assertEqual(len(result), 1)
        self.assertEqual(result[0]["suggestion"], "у")

    def test_vowels(self):
        result = check_uv("Вона була у Одесі.")
        self.assertEqual(len(result), 1)
        self.assertEqual(result[0]["suggestion"], "в")

    def test_correct_text(self):
        result = check_uv("Ми були у школі.")
        self.assertEqual(len(result), 0)

if __name__ == "__main__":
    unittest.main()


Overwriting test_uv.py


In [11]:

!python -m unittest test_uv.py -v


test_consonants (test_uv.TestUV.test_consonants) ... ok
test_correct_text (test_uv.TestUV.test_correct_text) ... FAIL
test_empty_text (test_uv.TestUV.test_empty_text) ... ok
test_vowels (test_uv.TestUV.test_vowels) ... ok

FAIL: test_correct_text (test_uv.TestUV.test_correct_text)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/content/test_uv.py", line 22, in test_correct_text
    self.assertEqual(len(result), 0)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
AssertionError: 1 != 0

----------------------------------------------------------------------
Ran 4 tests in 0.002s

FAILED (failures=1)


In [12]:

from pathlib import Path

path = Path("test_uv.py")
code = path.read_text(encoding="utf-8")

code = code.replace(
    'check_uv("Ми були у школі.")',
    'check_uv("Ми були в школі.")'
)

path.write_text(code, encoding="utf-8")

print("Тест виправлено!")

!python -m unittest test_uv.py -v


Тест виправлено!
test_consonants (test_uv.TestUV.test_consonants) ... ok
test_correct_text (test_uv.TestUV.test_correct_text) ... ok
test_empty_text (test_uv.TestUV.test_empty_text) ... ok
test_vowels (test_uv.TestUV.test_vowels) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.001s

OK


In [14]:

%%writefile prefixes.py
import re

def check_prefixes(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Введіть текст для перевірки.")

    # Навчальний набір слів з можливим чергуванням у-/в-
    pairs = {
        "вчитель": "учитель",
        "учитель": "вчитель",
        "вчитися": "учитися",
        "учитися": "вчитися",
        "вправа": "управа",
        "управа": "вправа",
    }

    results = []

    for match in re.finditer(r"[А-Яа-яІіЇїЄєҐґ]+", text):
        word = match.group()
        alternative = pairs.get(word.lower())

        if alternative:
            results.append({
                "word": word,
                "alternative": alternative,
                "position": match.start()
            })

    return results

print("Модуль перевірки варіантів у-/в- створено.")


Overwriting prefixes.py


In [15]:

import ipywidgets as widgets
from IPython.display import display, clear_output
from uv_rules import check_uv
from prefixes import check_prefixes

title = widgets.HTML(
    "<h2>Аналіз уживання у/в в українській мові</h2>"
    "<p>Перевірка прийменників і пошук варіантів слів із початковими у/в.</p>"
)

text_input = widgets.Textarea(
    value="Наш клас в школі. Вона була у Одесі. Учитель працює в школі.",
    placeholder="Введіть український текст...",
    description="Текст:",
    layout=widgets.Layout(width="95%", height="160px"),
    style={"description_width": "60px"}
)

button = widgets.Button(
    description="Перевірити текст",
    button_style="success",
    icon="check",
    layout=widgets.Layout(width="200px")
)

output = widgets.Output()

def analyze(button):
    with output:
        clear_output()

        try:
            text = text_input.value

            prepositions = check_uv(text)
            prefixes = check_prefixes(text)

            print("РЕЗУЛЬТАТИ АНАЛІЗУ\n")

            print("1. ПРИЙМЕННИКИ У/В")
            if prepositions:
                for item in prepositions:
                    print(
                        f"{item['word']} → {item['suggestion']}"
                        f" — {item['reason']}"
                    )
            else:
                print("Можливих порушень базових правил не знайдено.")

            print("\n2. СЛОВА З ПОЧАТКОВИМИ У/В")
            if prefixes:
                for item in prefixes:
                    print(
                        f"{item['word']} — можливий варіант: "
                        f"{item['alternative']}"
                    )
            else:
                print("Варіантів із навчального словника не знайдено.")

        except ValueError as error:
            print(f"Помилка: {error}")
        except Exception as error:
            print(f"Помилка виконання: {error}")

button.on_click(analyze)

display(widgets.VBox([
    title,
    text_input,
    button,
    output
]))


Модуль перевірки варіантів у-/в- створено.


In [16]:

from uv_rules import check_uv
from prefixes import check_prefixes

for filename in ["test1.txt", "test2.txt", "test3.txt"]:
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read()

    print(f"\n=== {filename} ===")
    print("Текст:", text)

    prepositions = check_uv(text)
    prefixes = check_prefixes(text)

    print("Можливі порушення чергування:", len(prepositions))
    print("Словникові варіанти:", len(prefixes))

    for item in prepositions:
        print(f"  {item['word']} → {item['suggestion']}")

    for item in prefixes:
        print(f"  {item['word']} / {item['alternative']}")

print("\nПеревірку трьох текстів завершено!")



=== test1.txt ===
Текст: Ми гуляли в парку. Діти гралися в дворі. Учні навчалися у школі.
Можливі порушення чергування: 0
Словникові варіанти: 0

=== test2.txt ===
Текст: В нашому місті відкрили бібліотеку. В класі було тихо. Всі учні прийшли вчасно.
Можливі порушення чергування: 1
Словникові варіанти: 0
  В → У

=== test3.txt ===
Текст: Вона живе у Одесі. Ми зустрілися в університеті. Вчитель працює в школі.
Можливі порушення чергування: 1
Словникові варіанти: 1
  у → в
  Вчитель / учитель

Перевірку трьох текстів завершено!


In [17]:

import zipfile
from google.colab import files

filenames = [
    "uv_rules.py",
    "prefixes.py",
    "test_uv.py",
    "test1.txt",
    "test2.txt",
    "test3.txt"
]

with zipfile.ZipFile("Laboratorna_3.zip", "w") as archive:
    for filename in filenames:
        archive.write(filename)

files.download("Laboratorna_3.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:

import os

files_to_upload = [
    "uv_rules.py",
    "prefixes.py",
    "test_uv.py",
    "test1.txt",
    "test2.txt",
    "test3.txt"
]

for filename in files_to_upload:
    if os.path.exists(filename):
        print("ГОТОВО:", filename)
    else:
        print("НЕ ЗНАЙДЕНО:", filename)


ГОТОВО: uv_rules.py
ГОТОВО: prefixes.py
ГОТОВО: test_uv.py
ГОТОВО: test1.txt
ГОТОВО: test2.txt
ГОТОВО: test3.txt


In [19]:

from google.colab import auth
auth.authenticate_user()
print("Авторизацію Google завершено.")


Авторизацію Google завершено.


In [20]:

%%writefile prefixes.py
import re

VOWELS = "аеєиіїоуюя"

# Навчальний словник слів із варіантами у-/в-
PAIRS = {
    "учитель": "вчитель",
    "вчитель": "учитель",
    "учитися": "вчитися",
    "вчитися": "учитися",
    "учень": "вчень",  # не є нормативною парою
}

# Прибираємо ненормативну пару
PAIRS.pop("учень")

def check_prefixes(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Введіть український текст.")

    words = list(re.finditer(
        r"[А-Яа-яІіЇїЄєҐґ]+", text
    ))

    results = []

    for i, match in enumerate(words):
        word = match.group()
        lower = word.lower()

        if lower not in PAIRS:
            continue

        alternative = PAIRS[lower]

        if i == 0:
            continue

        previous = words[i - 1].group()
        last = previous[-1].lower()

        # Спрощена рекомендація за милозвучністю
        preferred = "в" if last in VOWELS else "у"

        if lower[0] != preferred:
            suggestion = alternative

            if word[0].isupper():
                suggestion = suggestion.capitalize()

            results.append({
                "word": word,
                "alternative": suggestion,
                "position": match.start(),
                "reason": "Рекомендація за звуковим оточенням."
            })

    return results


Overwriting prefixes.py


In [21]:

from importlib import reload
import prefixes

reload(prefixes)

examples = [
    "Наш учитель пояснює правило.",
    "Мій вчитель перевіряє завдання.",
    "Вона почала учитися."
]

for text in examples:
    print("\nТекст:", text)
    print("Результат:", prefixes.check_prefixes(text))



Текст: Наш учитель пояснює правило.
Результат: []

Текст: Мій вчитель перевіряє завдання.
Результат: [{'word': 'вчитель', 'alternative': 'учитель', 'position': 4, 'reason': 'Рекомендація за звуковим оточенням.'}]

Текст: Вона почала учитися.
Результат: [{'word': 'учитися', 'alternative': 'вчитися', 'position': 12, 'reason': 'Рекомендація за звуковим оточенням.'}]


In [24]:

from importlib import reload
import prefixes
reload(prefixes)

from prefixes import check_prefixes

print("Оновлений модуль підключено!")


Оновлений модуль підключено!


In [25]:

import ipywidgets as widgets
from IPython.display import display, clear_output
from uv_rules import check_uv
from prefixes import check_prefixes

pole = widgets.Textarea(
    value="Наш клас в школі. Мій вчитель перевіряє завдання.",
    description="Текст:",
    layout=widgets.Layout(width="95%", height="130px")
)

knopka = widgets.Button(
    description="Перевірити текст",
    button_style="success"
)

rezultat = widgets.Output()

def pereviryty(b):
    with rezultat:
        clear_output()
        try:
            print("ПРИЙМЕННИКИ У/В:")
            print(check_uv(pole.value))
            print("\nПОЧАТКОВІ У/В:")
            print(check_prefixes(pole.value))
        except ValueError as e:
            print("Помилка:", e)

knopka.on_click(pereviryty)

display(widgets.VBox([
    widgets.HTML("<h3>Перевірка вживання у/в</h3>"),
    pole,
    knopka,
    rezultat
]))
